# Week 3 – PySpark Device-Level Aggregation
**Capstone: Smart Home Energy Usage Tracker**

## Setup

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, hour, date_format, round as spark_round, when

spark = SparkSession.builder.appName("SmartEnergyTracker").getOrCreate()
print("Spark session started")

Spark session started


## Load Dataset

In [3]:
from google.colab import files
uploaded = files.upload()

Saving energy_usage.csv to energy_usage.csv


In [4]:
df = spark.read.csv("energy_usage.csv", header=True, inferSchema=True)
df = df.withColumn("date", date_format(col("timestamp"), "yyyy-MM-dd"))
df = df.withColumn("hour", hour(col("timestamp")))

print("Total records:", df.count())
df.show()

Total records: 77
+------+---------+---------------+-------+------------+----------+-------------------+------+----------+----+
|log_id|device_id|    device_name|room_id|   room_name|energy_kwh|          timestamp|status|      date|hour|
+------+---------+---------------+-------+------------+----------+-------------------+------+----------+----+
|     1|     D001|Air Conditioner|   R001| Living Room|      1.82|2025-04-01 00:00:00|active|2025-04-01|   0|
|     2|     D002|   Refrigerator|   R002|     Kitchen|      0.45|2025-04-01 01:00:00|active|2025-04-01|   1|
|     3|     D003|Washing Machine|   R003|Utility Room|      0.91|2025-04-01 02:00:00|  idle|2025-04-01|   2|
|     4|     D004|     Television|   R001| Living Room|      0.12|2025-04-01 03:00:00|active|2025-04-01|   3|
|     5|     D005|      Microwave|   R002|     Kitchen|      0.08|2025-04-01 04:00:00|  idle|2025-04-01|   4|
|     6|     D006|   Water Heater|   R004|    Bathroom|       1.2|2025-04-01 05:00:00|active|2025-04-0

## Device-Level Energy Summary

In [5]:
device_summary = df.groupBy("device_id", "device_name") \
    .agg(
        sum("energy_kwh").alias("total_kwh"),
        avg("energy_kwh").alias("avg_kwh"),
        count("energy_kwh").alias("num_logs")
    ) \
    .withColumn("total_kwh", spark_round(col("total_kwh"), 3)) \
    .withColumn("avg_kwh",   spark_round(col("avg_kwh"),   3)) \
    .orderBy(col("total_kwh").desc())

print("Device-Level Energy Summary")
device_summary.show()

Device-Level Energy Summary
+---------+---------------+---------+-------+--------+
|device_id|    device_name|total_kwh|avg_kwh|num_logs|
+---------+---------------+---------+-------+--------+
|     D001|Air Conditioner|     19.5|   1.95|      10|
|     D010|    Room Heater|    10.68|  1.526|       7|
|     D006|   Water Heater|     9.68|   1.21|       8|
|     D003|Washing Machine|     5.43|  0.776|       7|
|     D007|     Dishwasher|     4.11|  0.587|       7|
|     D002|   Refrigerator|     4.03|  0.448|       9|
|     D004|     Television|     1.08|  0.135|       8|
|     D005|      Microwave|     0.69|  0.099|       7|
|     D009|    Ceiling Fan|      0.5|  0.071|       7|
|     D008| Laptop Charger|     0.35|   0.05|       7|
+---------+---------------+---------+-------+--------+



## Room-Level Energy Summary

In [6]:
room_summary = df.groupBy("room_id", "room_name") \
    .agg(
        sum("energy_kwh").alias("total_kwh"),
        avg("energy_kwh").alias("avg_kwh"),
        count("energy_kwh").alias("num_logs")
    ) \
    .withColumn("total_kwh", spark_round(col("total_kwh"), 3)) \
    .withColumn("avg_kwh",   spark_round(col("avg_kwh"),   3)) \
    .orderBy(col("total_kwh").desc())

print("Room-Level Energy Summary")
room_summary.show()

Room-Level Energy Summary
+-------+------------+---------+-------+--------+
|room_id|   room_name|total_kwh|avg_kwh|num_logs|
+-------+------------+---------+-------+--------+
|   R001| Living Room|    21.08|  0.843|      25|
|   R005|     Bedroom|    11.03|  0.788|      14|
|   R004|    Bathroom|     9.68|   1.21|       8|
|   R002|     Kitchen|     8.83|  0.384|      23|
|   R003|Utility Room|     5.43|  0.776|       7|
+-------+------------+---------+-------+--------+



## Peak vs Off-Peak Usage

In [7]:
peak_hours = [8, 9, 10, 18, 19, 20, 21]

df = df.withColumn(
    "usage_type",
    when(col("hour").isin(peak_hours), "peak").otherwise("off-peak")
)

peak_vs_offpeak = df.groupBy("device_name", "usage_type") \
    .agg(sum("energy_kwh").alias("total_kwh")) \
    .withColumn("total_kwh", spark_round(col("total_kwh"), 3)) \
    .orderBy("device_name", "usage_type")

print("Peak vs Off-Peak Usage per Device")
peak_vs_offpeak.show()

Peak vs Off-Peak Usage per Device
+---------------+----------+---------+
|    device_name|usage_type|total_kwh|
+---------------+----------+---------+
|Air Conditioner|  off-peak|     11.6|
|Air Conditioner|      peak|      7.9|
|    Ceiling Fan|  off-peak|     0.22|
|    Ceiling Fan|      peak|     0.28|
|     Dishwasher|  off-peak|     2.75|
|     Dishwasher|      peak|     1.36|
| Laptop Charger|  off-peak|     0.35|
|      Microwave|  off-peak|     0.59|
|      Microwave|      peak|      0.1|
|   Refrigerator|  off-peak|     3.59|
|   Refrigerator|      peak|     0.44|
|    Room Heater|  off-peak|      4.7|
|    Room Heater|      peak|     5.98|
|     Television|  off-peak|     1.08|
|Washing Machine|  off-peak|     2.73|
|Washing Machine|      peak|      2.7|
|   Water Heater|  off-peak|     9.68|
+---------------+----------+---------+



## High Consumption Detection

In [8]:
avg_kwh = df.agg(avg("energy_kwh").alias("avg")).collect()[0]["avg"]
high_consumption = df.filter(col("energy_kwh") > avg_kwh * 2)

print(f"High Consumption Logs (> 2x average of {avg_kwh:.3f} kWh)")
high_consumption.select("device_name", "room_name", "energy_kwh", "timestamp", "status").show()

High Consumption Logs (> 2x average of 0.728 kWh)
+---------------+-----------+----------+-------------------+------+
|    device_name|  room_name|energy_kwh|          timestamp|status|
+---------------+-----------+----------+-------------------+------+
|Air Conditioner|Living Room|      1.82|2025-04-01 00:00:00|active|
|    Room Heater|    Bedroom|       1.5|2025-04-01 09:00:00|active|
|Air Conditioner|Living Room|       1.9|2025-04-01 10:00:00|active|
|Air Conditioner|Living Room|       2.1|2025-04-02 00:00:00|active|
|    Room Heater|    Bedroom|      1.55|2025-04-02 09:00:00|active|
|Air Conditioner|Living Room|      1.75|2025-04-02 10:00:00|active|
|    Room Heater|    Bedroom|      1.48|2025-04-02 19:00:00|active|
|Air Conditioner|Living Room|      1.95|2025-04-03 00:00:00|active|
|    Room Heater|    Bedroom|       1.6|2025-04-03 07:00:00|active|
|Air Conditioner|Living Room|       2.2|2025-04-03 08:00:00|active|
|Air Conditioner|Living Room|      1.88|2025-04-04 00:00:00|active

In [10]:
from pyspark.sql.functions import sum as spark_sum

In [11]:
top_devices = df.groupBy("device_id") \
    .agg(spark_sum("energy_kwh").alias("total_energy")) \
    .orderBy(col("total_energy").desc()) \
    .limit(5)

print("Top 5 Energy-Consuming Devices")
top_devices.show()

top_devices.write.mode("overwrite").csv("top_devices_output", header=True)

Top 5 Energy-Consuming Devices
+---------+------------------+
|device_id|      total_energy|
+---------+------------------+
|     D001|19.499999999999996|
|     D010|             10.68|
|     D006|              9.68|
|     D003| 5.430000000000001|
|     D007| 4.109999999999999|
+---------+------------------+



## Save Output

In [12]:
device_summary.coalesce(1).write.csv("spark_device_output", header=True, mode="overwrite")
room_summary.coalesce(1).write.csv("spark_room_output",   header=True, mode="overwrite")
print("Outputs saved.")

Outputs saved.
